# 1.1 — AutoFT baseline on Google Colab (GPU testing modality)

Colab twin of `1.0-g30-autoft-baseline.ipynb`. Same native AutoFT reimplementation
(WILDS iWildCam leaderboard #1, arXiv:2401.10220), but sized for a Colab GPU so the
heavy stages (HPO outer loop + full fine-tune) are testable without the workstation's
8 GB RTX 4070 / full disk.

**Persistence model**

| What | Where | Why |
|---|---|---|
| Repo + code | `/content/...` (ephemeral) | cloned each session |
| iWildCam dataset | `/content/iwild_data` (ephemeral) | ~tens of GB extracted; free Google Drive (15 GB) cannot hold it — re-downloaded per session (~one coffee) |
| Checkpoints + reports | Google Drive via `models/` and `reports/` symlinks | survives disconnects; an overnight run's `final_model.pt` must not die with the VM |
| CLIP weights | Google Drive via `TORCH_HOME` | avoids re-downloading ViT-B/16 every session |

**Prerequisite — push your branch first.** The clone below fetches from GitHub, so the
AutoFT code must be committed and pushed (`git push -u origin <branch>`); set `GIT_REF`
to that branch/tag. Uncommitted local work is not visible to Colab.
Fully self-contained: the only inputs are this notebook, the public GitHub repo
(cloned below), and optionally a `WANDB_API_KEY` for metrics tracking — nothing
from the local workstation is needed. If the key is absent the pipeline still runs;
metrics fall back to loguru logs + the JSON artifacts on Drive.

In [ ]:
import os
import sys
from pathlib import Path

assert "google.colab" in sys.modules, "This notebook is written for Google Colab."

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

from google.colab import drive

drive.mount("/content/drive")

# --- configuration -----------------------------------------------------------
GIT_REF = "main"  # branch or tag that has the AutoFT code pushed
REPO_DIR = Path("/content/idl-f26-g30-project")
DRIVE_ROOT = Path("/content/drive/MyDrive/idl-f26/project")
DATA_DIR_SESSION = Path("/content/iwild_data")  # ephemeral, re-downloaded per session

for p in (DRIVE_ROOT / "models", DRIVE_ROOT / "reports", DRIVE_ROOT / "models" / "hub"):
    p.mkdir(parents=True, exist_ok=True)

# Read by iwild/config.py at import time: keep the dataset on the VM disk.
os.environ["IWILD_DATA_DIR"] = str(DATA_DIR_SESSION)
# Keep CLIP weight downloads on Drive so they persist across sessions.
os.environ["TORCH_HOME"] = str(DRIVE_ROOT / "models" / "hub")

print(f"dataset (session): {DATA_DIR_SESSION}")
print(f"artifacts (drive): {DRIVE_ROOT}")

NVIDIA L4, 23034 MiB
Mounted at /content/drive
dataset (session): /content/iwild_data
artifacts (drive): /content/drive/MyDrive/idl-f26/project


## 1. Install dependencies and clone the repo

Colab ships torch/torchvision with CUDA already; we add the training stack on top and
install the project editable (flit backend, same as the workstation `.venv`).

In [ ]:
%pip install -q open-clip-torch "wilds==2.0.0" optuna typer loguru python-dotenv tqdm wandb

if REPO_DIR.exists():
    !rm -rf {REPO_DIR}
!git clone --branch {GIT_REF} https://github.com/rnjema/idl-f26-g30-project.git {REPO_DIR}

%cd {REPO_DIR}
%pip install -q -e .

# Route artifacts to Drive for the rest of the session (repo dirs are empty save .gitkeep).
!rm -rf models reports
!ln -s {DRIVE_ROOT}/models models
!ln -s {DRIVE_ROOT}/reports reports
!ls -la models reports

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
Cloning into '/content/idl-f26-g30-project'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 72 (delta 26), reused 66 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 73.40 KiB | 10.49 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/idl-f26-g30-project
  Installing build dependencies ... done
  Checking if build 

**Metrics tracking (optional): Weights & Biases.** Add these Colab Secrets (same
values as in the project `.env`): `WANDB_API_KEY` (required for logging),
`WANDB_PROJECT` and `WANDB_ENTITY` (optional — they select the target project/team;
without them runs land in the fallback project `idl_f26_g30_autoft` under your personal
entity). The cell below copies all three into the environment, and every stage logs
scalars: per-trial HPO objectives + sampled loss weights, per-epoch ID-val F1 during
fine-tuning, and final split metrics. Without a key the stages run unchanged and only
write the loguru logs + JSON artifacts.

In [ ]:
WANDB = True  # set False to disable

try:
    from google.colab import userdata

    for _key in ("WANDB_API_KEY", "WANDB_PROJECT", "WANDB_ENTITY"):
        try:
            os.environ.setdefault(_key, userdata.get(_key))
        except Exception:
            pass  # secret not configured; fall back to os.environ / built-in defaults
except Exception:
    pass  # not on Colab

import wandb

if WANDB and not os.environ.get("WANDB_API_KEY"):
    WANDB = False
    print("No WANDB_API_KEY found; running without wandb (JSON artifacts still written).")
if WANDB:
    wandb.login()
    print(
        "wandb target: project",
        os.environ.get("WANDB_PROJECT", "idl_f26_g30_autoft (fallback)"),
        "| entity",
        os.environ.get("WANDB_ENTITY", "(personal default)"),
    )
wandb_flag = "--wandb" if WANDB else "--no-wandb"

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rnjema_cmu (team-support-vectors-idlf26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb target: project iwilds | entity team-support-vectors-idlf26


## 2. Session bootstrap: download + verify the dataset, build the FT CSV

~12 GB compressed, fetched directly from the CodaLab bundle (certificate-less
endpoint first, no-cert-verification https fallback — the bundle's TLS cert is
currently broken). Verification asserts the split sizes (129,809 / 14,961 / 42,791 /
7,314 / 8,154), the 182-class count, the camera-trap counts (243 / 32 / 48), and the
leakage rule that no trap crosses split slabs.

If the download fails, grab the bundle manually at
https://worksheets.codalab.org/bundles/0x6313da2b204647e79a14b468131fcd64 and place
the extracted `iwildcam_v2.0/` under `/content/iwild_data/raw/`.

In [ ]:
!{sys.executable} iwild/dataset.py download

2026-10-20 18:16:58.959 | INFO     | iwild.config:<module>:12 - PROJ_ROOT path is: /content/idl-f26-g30-project
2026-10-20 18:16:59.087 | INFO     | __main__:_download_and_extract:112 - Downloading iWildCam v2.0 from http://worksheets.codalab.org/rest/bundles/0x6313da2b204647e79a14b468131fcd64/contents/blob/ (~12 GB compressed)...
iwildcam_v2.0.tar.gz:  75% 8.97G/12.0G [14:05<04:35, 10.9MB/s]

In [ ]:
# FLYP/AutoFT fine-tuning CSV: one row per (train image x 2 text prompts) = 259,618 rows.
!{sys.executable} iwild/dataset.py make-ft-csv

# Leakage-free 80/20 camera-trap split of the official train split (selector lane; cheap).
!{sys.executable} iwild/dataset.py location-split --seed 42

## 3. Zero-shot CLIP model (ViT-B/16)

Builds the 182-way zero-shot head from the two iWildCam prompts per class (rows of
`labels.csv` with y < 99999, matching the official auto-ft code and WILDS v2.0's 182 classes).
Saved to Drive via the `models/` symlink, so this step is a no-op after the first session.

In [ ]:
!{sys.executable} iwild/modeling/train.py zeroshot

## 4. Sanity: zero-shot macro F1 on the OOD val split

A healthy ViT-B/16 zero-shot model should land in the tens-of-percent macro F1 range.
Anything near zero almost always means a label/prompt mismatch, not a bad model.

In [ ]:
import json

from iwild.config import REPORTS_DIR, ZEROSHOT_DIR

ckpt = ZEROSHOT_DIR / "clip_vitb16_iwildcam.pt"
!{sys.executable} iwild/modeling/train.py eval --ckpt {ckpt} {wandb_flag} --wandb-run-name zeroshot_sanity

report = json.loads((REPORTS_DIR / f"autoft_eval_{ckpt.stem}.json").read_text())
print(json.dumps(report, indent=2))

## 5. AutoFT run

One command runs the Optuna outer loop (objective = macro F1 on 1,000 class-balanced OOD-val
examples) and then the final full fine-tune (best epoch by ID-val macro F1). Checkpoints and
per-trial JSONs land in `models/autoft/<run-name>/` — i.e. on Drive.

- `SMOKE = True` → 3 trials x 5 inner steps, batch 16, 2,000 train rows: minutes on any GPU.
- `SMOKE = False` → downscaled reproduction: 100 trials x 10 inner steps, batch 64 x accum 2.
  The paper's official recipe is 500 evals x 100 inner steps at batch 128-256 (A100-class,
  several hours just for the outer loop); on a free T4 prefer `--batch-size 32
  --accumulation-steps 4`. Trial 0 is seeded with the paper's pre-learned iWildCam weights
  (`autoft_hparams_iwildcam.json`), so even a short search starts from a sensible point.

In [ ]:
SMOKE = False
RUN_NAME = "smoke" if SMOKE else "autoft_vitb16_trials"

flags = "--smoke --no-init-hparams" if SMOKE else "--hopt-evals 100 --inner-steps 10"
!{sys.executable} iwild/modeling/train.py autoft --run-name {RUN_NAME} {flags} {wandb_flag}

2026-10-20 18:46:03.103 | INFO     | iwild.config:<module>:12 - PROJ_ROOT path is: /content/idl-f26-g30-project
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rnjema_cmu (team-support-vectors-idlf26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run gs8c131w (0.2s)
wandb: ⣯ setting up run gs8c131w (0.2s)
wandb: ⣟ setting up run gs8c131w (0.2s)
wandb: ⡿ setting up run gs8c131w (0.2s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/idl-f26-g30-project/wandb/run-20261020_184610-gs8c131w
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run autoft_autoft_vitb16_trials
wandb: ⭐️ View project at https://wandb.ai/team

## 6. FLYP comparison run (the baseline AutoFT improves on)

Same backbone, same data stream, fixed loss weights ce=1 / flyp=1, no hyperparameter search.

In [ ]:
flyp_name = "flyp_smoke" if SMOKE else "flyp_baseline"
flyp_flags = "--smoke" if SMOKE else ""
!{sys.executable} iwild/modeling/train.py flyp --run-name {flyp_name} {flyp_flags} {wandb_flag}

## 7. Final evaluation (OOD test touched once)

Report macro F1 / accuracy on IDVal / IDTest / OODVal / OODTest. Per the research protocol,
all threshold/hyperparameter decisions come from the ID val or the official OOD val split;
the OOD test numbers below are the once-only headline.

In [ ]:
final_ckpt = Path(f"models/autoft/{RUN_NAME}/final_model.pt")
assert final_ckpt.exists(), "run section 5 first"
!{sys.executable} iwild/modeling/train.py eval --ckpt {final_ckpt} {wandb_flag}

(REPORTS_DIR / f"autoft_eval_{final_ckpt.stem}.json")

## 8. Optional: WiSE-FT interpolation (the leaderboard ensembling trick)

The 52.0 leaderboard entry interpolates fine-tuned and zero-shot weights
(`theta = (1-alpha) * theta_0 + alpha * theta_1`), selecting alpha by ID-val macro F1.
AutoFT without ensembling gains +2.7 macro F1 over FLYP; ensembling adds the rest.

In [ ]:
if not SMOKE:
    !{sys.executable} iwild/modeling/train.py wisefmt --finetuned {final_ckpt} --run-name wisefmt_{RUN_NAME}

## 9. Context: what to compare against

Leaderboard numbers (official WILDS protocol, CLIP **ViT-L/14@336**):

| Method | OOD test macro F1 |
|---|---|
| AutoFT (WiSE-FT ensembled) | **52.0** |
| AutoFT (plain) | ~48.7 |
| FLYP | 46.0 |

This notebook reproduces the method on **ViT-B/16** (compute decision per the research
plan), so absolute numbers will sit well below 46 — that is expected and fine. What matters
for the project is the *relative* AutoFT-vs-FLYP gap on our backbone, and having the frozen
classifier that the CF-delta abstention selector is built on top of.

## 10. Session resume cheat sheet

- Everything persistent lives in `MyDrive/idl_f26_g30_project/{models,reports}` — reconnect,
  re-run sections 0–1, then jump straight to section 5 with `SMOKE = False`.
- The dataset and FT CSV are session-local: re-run sections 2 (about one coffee) after any
  disconnect. Upgrading to Drive storage >40 GB lets you point `IWILD_DATA_DIR` at Drive
  instead and skip this.
- Free-tier limits: ~12 h runtime per session with a T4. A downscaled run (100 evals x 10
  inner steps, 20 epochs final FT at batch 64) fits comfortably; the full official recipe
  wants Colab Pro (A100) or a cluster.